# Anima 出图 + LoRA 训练(Colab 免费 T4 版)

模型:[circlestone-labs/Anima](https://huggingface.co/circlestone-labs/Anima)(Cosmos-Predict2 2B DiT + Qwen3 0.6B + Qwen-Image VAE)  
训练器:[kohya-ss/sd-scripts](https://github.com/kohya-ss/sd-scripts) 的 `anima_train_network.py`

**使用方法**:菜单 `代码执行程序 → 更改运行时类型 → T4 GPU`,然后从上往下依次运行。

> ⚠️ Anima 许可证为 CircleStone Labs 非商业许可,LoRA 同样限于非商业用途。  
> ⚠️ 这份 notebook 没有在真实 T4 上跑过。T4 不原生支持 bf16,第 5 步如果 loss 出现 NaN,请看那一格里的说明。

## 0. 检查 GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| bf16 supported:", torch.cuda.is_bf16_supported())

## 1. 安装 sd-scripts

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git
%cd /content/sd-scripts
!pip install -q -r requirements.txt
!pip install -q bitsandbytes huggingface_hub hf_transfer

## 2. 下载模型
如果 HF 要求登录,在 Colab 左侧「密钥」里添加 `HF_TOKEN`。

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

REPO = "circlestone-labs/Anima"
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# 按文件名在仓库里查找,避免路径写死
files = HfApi().list_repo_files(REPO)
def find(name):
    hits = [f for f in files if f.endswith(name)]
    assert hits, f"{name} not found in {REPO}: {files}"
    return hits[0]

DIT  = hf_hub_download(REPO, find("anima-base-v1.0.safetensors"), local_dir=MODEL_DIR)
QWEN = hf_hub_download(REPO, find("qwen_3_06b_base.safetensors"), local_dir=MODEL_DIR)
VAE  = hf_hub_download(REPO, find("qwen_image_vae.safetensors"), local_dir=MODEL_DIR)
print(DIT, QWEN, VAE, sep="\n")

## 3. 试出图(不带 LoRA)

提示词写法(来自模型主页):标签全小写、空格代替下划线;画师名前加 `@`;推荐前缀 `masterpiece, best quality, score_7, safe, `。  
Base 版推荐 30–50 步、CFG 4–5。`flow_shift` 按架构图里的 shift=3 设置。

In [ ]:
PROMPT = "masterpiece, best quality, score_7, safe, 1girl, solo, silver hair, blue eyes, school uniform, standing, cherry blossoms, looking at viewer"
NEG    = "worst quality, low quality, score_1, score_2, score_3, artist name, blurry, jpeg artifacts, chromatic aberration"

import os, glob
from IPython.display import display
from PIL import Image
os.makedirs("/content/out", exist_ok=True)

def generate(prompt, out="/content/out/test.png", lora=None, mult=1.0, seed=42, size=(1024, 1024), steps=30, cfg=4.5):
    cmd = f'''python anima_minimal_inference.py \\
      --dit "{DIT}" --vae "{VAE}" --text_encoder "{QWEN}" \\
      --prompt "{prompt}" --negative_prompt "{NEG}" \\
      --image_size {size[0]} {size[1]} --infer_steps {steps} --guidance_scale {cfg} \\
      --flow_shift 3.0 --seed {seed} --save_path "{out}" \\
      --qwen_image_vae_2d --text_encoder_cpu'''
    if lora:
        cmd += f' --lora_weight "{lora}" --lora_multiplier {mult}'
    get_ipython().system(cmd)

%cd /content/sd-scripts
generate(PROMPT)

In [ ]:
# 该脚本会把图保存到 --save_path 所在目录(文件名可能带时间戳/序号),这里直接展示最新的
imgs = sorted(glob.glob("/content/out/*.png"), key=os.path.getmtime)
if imgs: display(Image.open(imgs[-1]))
else: print("没有找到输出图,请查看上一格的日志")

## 4. 准备训练数据

数据格式:一个文件夹,里面是 `xxx.png/jpg` + 同名 `xxx.txt`(caption)。  
Anima 的 caption 是 **Danbooru 风格标签**(小写、空格分隔、逗号隔开),也可以混写自然语言。  
建议 15–50 张图起步,风格 LoRA 通常需要更多。

**方式 A(推荐)**:把数据放到 Google Drive 的 `MyDrive/anima_dataset/<名字>/`,运行下面的挂载。  
**方式 B**:直接把文件拖进 Colab 左侧文件栏的 `/content/dataset/`(会话结束就没了)。

In [ ]:
USE_DRIVE = True
DRIVE_DATASET = "/content/drive/MyDrive/anima_dataset/my_lora"   # ← 改成你的数据目录
DATASET_DIR = "/content/dataset"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !rm -rf {DATASET_DIR}
    !cp -r "{DRIVE_DATASET}" {DATASET_DIR}
else:
    os.makedirs(DATASET_DIR, exist_ok=True)

exts = (".png", ".jpg", ".jpeg", ".webp")
imgs = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(exts)]
missing = [f for f in imgs if not os.path.exists(os.path.join(DATASET_DIR, os.path.splitext(f)[0] + ".txt"))]
print(f"图片 {len(imgs)} 张,缺 caption {len(missing)} 张")
assert imgs, "数据目录里没有图片"

In [ ]:
# 可选:给缺 caption 的图补一个统一的触发词(只适合单一角色/画风,效果不如逐张打标)
TRIGGER = "my_style"
for f in missing:
    open(os.path.join(DATASET_DIR, os.path.splitext(f)[0] + ".txt"), "w").write(TRIGGER)
print("补全:", len(missing))

## 5. 开始训练

T4 参数说明:
- 分辨率 512、batch 1、`AdamW8bit`、`gradient_checkpointing`,并缓存 latent 和文本编码输出;
- 学习率 5e-5(模型主页建议 LoRA 用较低学习率,rank 32 从 2e-5 起;sd-scripts 文档示例是 1e-4)。过拟合/欠拟合时再调;
- 不训练 LLM Adapter(默认就不训练,主页明确说它容易被训坏);
- 显存不够就把 `BLOCKS_TO_SWAP` 调大(最大 26),会变慢;
- **bf16**:T4 没有原生 bf16。默认先用 `bf16`,如果报错或 loss=NaN,把 `PRECISION` 改成 `"fp16"` 再试(fp16 对这个模型是否稳定我没有验证)。

In [ ]:
RUN_NAME   = "my_anima_lora"
RESOLUTION = 512
RANK       = 16
LR         = "5e-5"
EPOCHS     = 10
REPEATS    = 10          # 每张图每个 epoch 重复次数;图少就调大
BLOCKS_TO_SWAP = 0       # OOM 时改成 8~20
PRECISION  = "bf16"      # 出问题改 "fp16"
OUT_DIR    = "/content/output"

os.makedirs(OUT_DIR, exist_ok=True)
toml = f'''
[general]
shuffle_caption = false
caption_extension = ".txt"

[[datasets]]
resolution = {RESOLUTION}
batch_size = 1
enable_bucket = true
min_bucket_reso = 384
max_bucket_reso = 768
bucket_reso_steps = 64

  [[datasets.subsets]]
  image_dir = "{DATASET_DIR}"
  num_repeats = {REPEATS}
'''
open("/content/dataset.toml", "w").write(toml)

swap = f"--blocks_to_swap={BLOCKS_TO_SWAP} --block_swap_h2d_only" if BLOCKS_TO_SWAP > 0 else ""
%cd /content/sd-scripts
!accelerate launch --num_cpu_threads_per_process 1 --mixed_precision {PRECISION} anima_train_network.py \
  --pretrained_model_name_or_path="{DIT}" \
  --qwen3="{QWEN}" \
  --vae="{VAE}" \
  --dataset_config="/content/dataset.toml" \
  --output_dir="{OUT_DIR}" --output_name="{RUN_NAME}" \
  --save_model_as=safetensors \
  --network_module=networks.lora_anima --network_dim={RANK} --network_alpha={RANK} \
  --network_train_unet_only \
  --learning_rate={LR} --optimizer_type="AdamW8bit" --lr_scheduler="constant" \
  --timestep_sampling="sigmoid" --discrete_flow_shift=1.0 \
  --max_train_epochs={EPOCHS} --save_every_n_epochs=1 \
  --mixed_precision="{PRECISION}" \
  --gradient_checkpointing \
  --cache_latents --cache_text_encoder_outputs \
  --qwen_image_vae_2d \
  --seed=42 {swap}

## 6. 用训练好的 LoRA 出图,并备份到 Drive

In [ ]:
lora_files = sorted(glob.glob(f"{OUT_DIR}/{RUN_NAME}*.safetensors"))
print(lora_files)
LORA = lora_files[-1]
%cd /content/sd-scripts
generate(PROMPT + ", my_style", out="/content/out/with_lora.png", lora=LORA, mult=1.0)
imgs = sorted(glob.glob("/content/out/with_lora*.png"), key=os.path.getmtime)
if imgs: display(Image.open(imgs[-1]))

In [ ]:
# 备份 LoRA 到 Drive(Colab 会话结束后本地文件会消失)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/anima_lora_out
!cp {OUT_DIR}/*.safetensors /content/drive/MyDrive/anima_lora_out/
!ls -la /content/drive/MyDrive/anima_lora_out/